# Label CafeF 2023 With DeepSeek V4 Flash

Notebook này đọc 1.000 dòng đầu từ `data/raw/cafef_raw/cafef_news_2023.csv`, nạp `system_prompt.txt` và `few_shots.txt`, rồi gọi DeepSeek Chat Completions để trích xuất sự kiện tài chính theo schema JSON.

Cách xử lý few-shot trong notebook này:
- `system_prompt.txt` giữ toàn bộ schema/quy tắc ổn định trong role `system`.
- `few_shots.txt` được đưa vào một message `user` ổn định ngay trước bài báo hiện tại. Cách này hợp với file few-shot đang là markdown/rút gọn, và giúp phần prefix lặp lại dễ được context cache.
- Bài báo hiện tại nằm ở message `user` cuối cùng, mỗi request chỉ xử lý một bài để tránh lẫn entity/event giữa các bài.
- API bật `response_format={"type": "json_object"}` để ép output là JSON hợp lệ theo hướng dẫn DeepSeek.

Nguồn API chính thức:
- Chat completions: https://api-docs.deepseek.com/api/create-chat-completion
- JSON output: https://api-docs.deepseek.com/guides/json_mode
- Models/pricing/cache fields: https://api-docs.deepseek.com/quick_start/pricing


In [13]:
from __future__ import annotations

import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent

DATA_PATH = ROOT / "data/raw_news/merged_news.csv"
SYSTEM_PROMPT_PATH = ROOT / "src/LLM_labeling/system_prompt.txt"
FEW_SHOTS_PATH = ROOT / "src/LLM_labeling/few_shots.txt"
OUTPUT_PATH = ROOT / "data/processed/10k_labeled_news.jsonl"

MODEL = "deepseek-v4-flash"
BASE_URL = "https://api.deepseek.com"
N_ROWS = 20000
REQUEST_SLEEP_SECONDS = 0.15
MAX_TOKENS = 8192
TEMPERATURE = 0
DISABLE_THINKING = True

print("ROOT:", ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())
print("SYSTEM_PROMPT_PATH exists:", SYSTEM_PROMPT_PATH.exists())
print("FEW_SHOTS_PATH exists:", FEW_SHOTS_PATH.exists())


ROOT: /Users/buithianhdao/SE365
DATA_PATH exists: True
SYSTEM_PROMPT_PATH exists: True
FEW_SHOTS_PATH exists: True


In [14]:
system_prompt = SYSTEM_PROMPT_PATH.read_text(encoding="utf-8").strip()
few_shots = FEW_SHOTS_PATH.read_text(encoding="utf-8").strip()

#Randomly sample N_ROWS from the dataset to avoid hitting the API limit
df = pd.read_csv(DATA_PATH).sample(n=N_ROWS, random_state=42).copy()

df["article_id"] = df["article_id"].astype(str)

print("Rows:", len(df))
print("System prompt chars:", len(system_prompt))
print("Few-shot chars:", len(few_shots))
display(df[["article_id", "title", "summary", "published_date", "url"]].head(3))


Rows: 20000
System prompt chars: 15124
Few-shot chars: 8440


,article_id,title,summary,published_date,url
22065,188251007222809692,UBCKNN phát cảnh báo nóng đến những nhà đầu tư...,"UBCKNN khuyến cáo nhà đầu tư thận trọng, đồng ...",2025-10-08,https://cafef.vn/ubcknn-phat-canh-bao-nong-den...
20318,188251225225755762,Lịch sự kiện và tin vắn chứng khoán ngày 26/12...,Tổng hợp toàn bộ tin vắn nổi bật liên quan đến...,2025-12-26,https://cafef.vn/lich-su-kien-va-tin-van-chung...
9527,1184617,Theo dấu dòng tiền cá mập 26/04: Khối ngoại mu...,"Phiên ngày 26/04, khối tự doanh công ty chứng ...",2024-04-26,https://vietstock.vn/2024/04/theo-dau-dong-tie...


## Prompt Builders

`few_shots.txt` hiện là markdown ví dụ rút gọn, không phải các cặp `user`/`assistant` hoàn chỉnh. Vì vậy notebook dùng nó như một khối tham khảo ổn định. Nếu sau này bạn tạo few-shot đầy đủ dạng `input/output`, có thể đổi sang alternating messages: `user(example_input)`, `assistant(example_output)`, rồi mới tới bài thật.

In [3]:
def clean_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def build_article_prompt(row: pd.Series) -> str:
    return f"""Bây giờ hãy trích xuất bài báo sau theo đúng schema JSON trong system prompt.

Yêu cầu bắt buộc:
- Chỉ trả về JSON hợp lệ, không markdown, không giải thích.
- `doc_id` nếu cần thì dùng article_id bên dưới.
- Nếu không có sự kiện tài chính rõ ràng: events = [] và main_topic = "other".

article_id: {clean_text(row.get('article_id'))}
source: {clean_text(row.get('source'))}
category: {clean_text(row.get('category'))}

TIÊU ĐỀ:
{clean_text(row.get('title'))}

TÓM TẮT:
{clean_text(row.get('summary'))}

NỘI DUNG:
{clean_text(row.get('content'))}
"""


def build_messages(row: pd.Series, include_few_shots: bool = True) -> list[dict[str, str]]:
    messages = [{"role": "system", "content": system_prompt}]
    if include_few_shots and few_shots:
        messages.append({
            "role": "user",
            "content": (
                "Dưới đây là các ví dụ mẫu. Hãy học cách chọn event_type, main_topic, "
                "cách điền attributes/context/evidence_text. Không trích xuất lại các ví dụ này.\n\n"
                f"{few_shots}"
            ),
        })
    messages.append({"role": "user", "content": build_article_prompt(row)})
    return messages


def extract_json_object(text: str) -> dict[str, Any]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise


preview_messages = build_messages(df.iloc[0])
print("Number of messages:", len(preview_messages))
for i, msg in enumerate(preview_messages):
    print(i, msg["role"], len(msg["content"]), msg["content"][:160].replace("\n", " "))


Number of messages: 3
0 system 15124 Bạn là hệ thống trích xuất sự kiện tài chính chuyên nghiệp cho thị trường chứng khoán Việt Nam. Nhiệm vụ của bạn là đọc MỘT bài báo tài chính tiếng Việt và tríc
1 user 8590 Dưới đây là các ví dụ mẫu. Hãy học cách chọn event_type, main_topic, cách điền attributes/context/evidence_text. Không trích xuất lại các ví dụ này.  ### Ví dụ 
2 user 1618 Bây giờ hãy trích xuất bài báo sau theo đúng schema JSON trong system prompt.  Yêu cầu bắt buộc: - Chỉ trả về JSON hợp lệ, không markdown, không giải thích. - `


In [9]:
def deepseek_chat(messages: list[dict[str, str]], *, max_retries: int = 3) -> dict[str, Any]:
    api_key = os.environ.get("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError("Missing DEEPSEEK_API_KEY. Set it before running API cells.")

    payload: dict[str, Any] = {
        "model": MODEL,
        "messages": messages,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "response_format": {"type": "json_object"},
        "stream": False,
    }
    if DISABLE_THINKING:
        payload["thinking"] = {"type": "disabled"}

    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(
        f"{BASE_URL}/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(req, timeout=180) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            error_body = e.read().decode("utf-8", errors="replace")
            if e.code in {429, 500, 502, 503, 504} and attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise RuntimeError(f"DeepSeek HTTP {e.code}: {error_body}") from e
        except urllib.error.URLError:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise

    raise RuntimeError("DeepSeek request failed after retries")


def label_one(row: pd.Series, include_few_shots: bool = True) -> dict[str, Any]:
    messages = build_messages(row, include_few_shots=include_few_shots)
    response = deepseek_chat(messages)
    choice = response["choices"][0]
    content = choice["message"].get("content") or ""

    parsed: dict[str, Any] | None = None
    parse_error: str | None = None
    try:
        parsed = extract_json_object(content)
        parsed.setdefault("doc_id", clean_text(row.get("article_id")))
    except json.JSONDecodeError as e:
        # Keep raw_response/usage so malformed JSON can be inspected or repaired later.
        parse_error = repr(e)

    return {
        "article_id": clean_text(row.get("article_id")),
        "title": clean_text(row.get("title")),
        "content": clean_text(row.get("content")),
        "summary": clean_text(row.get("summary")),
        "label": parsed,
    }


# Nhập DeepSeek API key

In [4]:
import os
import getpass

os.environ.pop("DEEPSEEK_API_KEY", None)
os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Nhập lại DeepSeek API key: ").strip()


## Test 1 Article

Chạy cell này trước để xem API key, JSON mode, prompt và schema có ổn không. Để tránh gọi API ngoài ý muốn, mặc định `RUN_ONE_API_CALL = False`.

In [7]:
RUN_ONE_API_CALL = False

if RUN_ONE_API_CALL:
    one_result = label_one(df.iloc[0], include_few_shots=True)
    print(json.dumps(one_result["label"], ensure_ascii=False, indent=2))
    print("usage:", one_result["usage"])
else:
    print("Dry run only. Set RUN_ONE_API_CALL = True to call DeepSeek for the first article.")
    print(build_article_prompt(df.iloc[0])[:1200])


Dry run only. Set RUN_ONE_API_CALL = True to call DeepSeek for the first article.
Bây giờ hãy trích xuất bài báo sau theo đúng schema JSON trong system prompt.

Yêu cầu bắt buộc:
- Chỉ trả về JSON hợp lệ, không markdown, không giải thích.
- `doc_id` nếu cần thì dùng article_id bên dưới.
- Nếu không có sự kiện tài chính rõ ràng: events = [] và main_topic = "other".

article_id: 188251007222809692
source: cafef
category: Thị trường chứng khoán

TIÊU ĐỀ:
UBCKNN phát cảnh báo nóng đến những nhà đầu tư nộp tiền vào tài khoản chứng khoán này

TÓM TẮT:
UBCKNN khuyến cáo nhà đầu tư thận trọng, đồng thời kiểm tra, đối chiếu kỹ thông tin pháp lý trước khi mở tài khoản hay tham gia các hoạt động đầu tư để tránh bị lợi dụng, lừa đảo.

NỘI DUNG:
Ủy ban Chứng khoán Nhà nước (UBCKNN) vừa phát đi cảnh báo liên quan đến dấu hiệu giả mạo, lừa đảo mang tên Công ty TNHH Đầu tư Tư vấn STICCapital .
Theo UBCKNN, cơ quan này đã nhận được báo cáo từ Văn phòng đại diện Công ty đầu tư STIC Investments, Inc. tại

In [18]:
print(df.iloc[0]['content'] )


Ủy ban Chứng khoán Nhà nước (UBCKNN) vừa phát đi cảnh báo liên quan đến dấu hiệu giả mạo, lừa đảo mang tên Công ty TNHH Đầu tư Tư vấn STICCapital .
Theo UBCKNN, cơ quan này đã nhận được báo cáo từ Văn phòng đại diện Công ty đầu tư STIC Investments, Inc. tại Việt Nam, phản ánh việc một số nhà đầu tư bị hướng dẫn nạp tiền vào tài khoản chứng khoán mang tên Công ty TNHH Đầu tư Tư vấn STICCapital (số tài khoản 119003023002 tại Ngân hàng VietinBank).
Đáng chú ý, đối tượng còn thiết lập website tại địa chỉ https://sticinv.com và ứng dụng di động STCS Pro trên App Store và Google Play để tạo vỏ bọc hoạt động đầu tư hợp pháp.﻿
﻿UBCKNN khẳng định chưa từng cấp phép hoạt động hoặc bất kỳ giấy phép nào liên quan đến pháp nhân có tên Công ty TNHH Đầu tư Tư vấn STICCapital.
Cơ quan này khuyến cáo nhà đầu tư thận trọng, đồng thời kiểm tra, đối chiếu kỹ thông tin pháp lý trước khi mở tài khoản hay tham gia các hoạt động đầu tư để tránh bị lợi dụng, lừa đảo.


## Batch Label 10000 Rows

Output được append vào JSONL. Nếu notebook dừng giữa chừng, chạy lại cell này sẽ bỏ qua `article_id` đã có trong output.

In [5]:
def run_label_batch(start_row: int, end_row: int, run_batch: bool = True):
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    processed_ids: set[str] = set()

    # Chỉ skip các article_id đã label thành công. Các dòng lỗi sẽ được retry.
    if OUTPUT_PATH.exists():
        with OUTPUT_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    if record.get("label") is not None and not record.get("error"):
                        processed_ids.add(str(record.get("article_id")))

    batch_df = df.iloc[start_row:end_row].copy()
    todo = batch_df.loc[~batch_df["article_id"].astype(str).isin(processed_ids)].copy()

    print("Output:", OUTPUT_PATH)
    print("Rows:", start_row, "to", end_row)
    print("Batch size:", len(batch_df))
    print("Already processed in this batch:", len(batch_df) - len(todo))
    print("Todo:", len(todo))

    if run_batch:
        with OUTPUT_PATH.open("a", encoding="utf-8") as out:
            for n, (_, row) in enumerate(todo.iterrows(), start=1):
                article_id = clean_text(row.get("article_id"))

                try:
                    result = label_one(row, include_few_shots=True)
                    result.setdefault("error", None)
                except Exception as e:
                    result = {
                        "article_id": article_id,
                        "title": clean_text(row.get("title")),
                        "published_date": clean_text(row.get("published_date")),
                        "url": clean_text(row.get("url")),
                        "label": None,
                        "raw_response": None,
                        "finish_reason": None,
                        "usage": {},
                        "model": MODEL,
                        "error": repr(e),
                    }

                out.write(json.dumps(result, ensure_ascii=False) + "\n")
                out.flush()

                if n % 10 == 0 or result.get("error"):
                    print(f"{n}/{len(todo)} article_id={article_id} error={result.get('error')}")

                time.sleep(REQUEST_SLEEP_SECONDS)
    else:
        print("Dry run only. Set run_batch=True to label this batch.")


In [27]:
run_label_batch(0, 1000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 0 to 1000
Batch size: 1000
Already processed in this batch: 238
Todo: 762
10/762 article_id=1200167 error=None
20/762 article_id=188250318210854194 error=None
30/762 article_id=188230910221303299 error=None
40/762 article_id=188250929200751963 error=None
50/762 article_id=188250521134226799 error=None
60/762 article_id=1279297 error=None
70/762 article_id=1376908 error=None
80/762 article_id=188251214182839882 error=None
90/762 article_id=188250603113356832 error=None
100/762 article_id=188250513221611983 error=None
110/762 article_id=188260507140145529 error=None
120/762 article_id=188241028113515356 error=None
130/762 article_id=188240914131717729 error=None
140/762 article_id=188231106081527028 error=None
150/762 article_id=188251211114020385 error=None
160/762 article_id=188240925181443482 error=None
170/762 article_id=188240718163620257 error=None
180/762 article_id=859736 error=None
190/762 article_id=1

In [28]:
run_label_batch(1000, 2000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 1000 to 2000
Batch size: 1000
Already processed in this batch: 3
Todo: 997
10/997 article_id=188230828113526329 error=None
20/997 article_id=188250728031314636 error=None
30/997 article_id=820630 error=None
40/997 article_id=1052262 error=None
50/997 article_id=188251123215125451 error=None
60/997 article_id=1448917 error=None
70/997 article_id=188230511143405604 error=None
80/997 article_id=18826040119204519 error=None
90/997 article_id=188250506110310059 error=None
100/997 article_id=188241103195514793 error=None
110/997 article_id=188251105190041039 error=None
120/997 article_id=188260502204726838 error=None
130/997 article_id=1356480 error=None
140/997 article_id=820845 error=None
150/997 article_id=188250409121222994 error=None
160/997 article_id=188260128110608965 error=None
170/997 article_id=188250711110346618 error=None
180/997 article_id=188250427134550429 error=None
190/997 article_id=865644 error=

In [29]:
run_label_batch(2000, 3000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 2000 to 3000
Batch size: 1000
Already processed in this batch: 6
Todo: 994
10/994 article_id=188230910225358556 error=None
20/994 article_id=188230814134351172 error=None
30/994 article_id=1267918 error=None
40/994 article_id=188250616152710145 error=None
50/994 article_id=188240429170450375 error=None
60/994 article_id=1117620 error=None
70/994 article_id=188260506111705379 error=None
80/994 article_id=1237315 error=None
90/994 article_id=1369108 error=None
100/994 article_id=188240123111224833 error=None
110/994 article_id=1184923 error=None
120/994 article_id=1363904 error=None
121/994 article_id=188240129223735689 error=TimeoutError('The read operation timed out')
130/994 article_id=188250530113629549 error=None
140/994 article_id=188240509155033219 error=None
150/994 article_id=1882402162342353 error=None
160/994 article_id=869517 error=None
170/994 article_id=188250702225444365 error=None
180/994 articl

In [30]:
run_label_batch(3000, 4000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 3000 to 4000
Batch size: 1000
Already processed in this batch: 14
Todo: 986
10/986 article_id=188251017110741123 error=None
20/986 article_id=838807 error=None
30/986 article_id=892291 error=None
40/986 article_id=1222773 error=None
50/986 article_id=983251 error=None
60/986 article_id=1138327 error=None
70/986 article_id=188241016213559975 error=None
80/986 article_id=188240305152616248 error=None
90/986 article_id=188240804083447758 error=None
100/986 article_id=188241009141024262 error=None
110/986 article_id=188240829164301925 error=None
120/986 article_id=188240903213834599 error=None
130/986 article_id=188251022152524875 error=None
140/986 article_id=188240804161105757 error=None
150/986 article_id=20230217182933023 error=None
160/986 article_id=188260313193922121 error=None
170/986 article_id=188240416104803164 error=None
180/986 article_id=188250415220008569 error=None
190/986 article_id=1882501271519

In [31]:
run_label_batch(4000, 5000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 4000 to 5000
Batch size: 1000
Already processed in this batch: 29
Todo: 971
10/971 article_id=852537 error=None
20/971 article_id=1019016 error=None
30/971 article_id=1050809 error=None
40/971 article_id=188260119222318398 error=None
50/971 article_id=188250611132019469 error=None
60/971 article_id=188251106165144208 error=None
70/971 article_id=1228314 error=None
80/971 article_id=886007 error=None
90/971 article_id=826757 error=None
100/971 article_id=188260319102640951 error=None
110/971 article_id=188240808225920582 error=None
120/971 article_id=950318 error=None
130/971 article_id=1201638 error=None
140/971 article_id=188250410145639046 error=None
150/971 article_id=188260519154458003 error=None
160/971 article_id=188230920065727469 error=None
170/971 article_id=188241017204313 error=None
180/971 article_id=188230701080927385 error=None
190/971 article_id=1059798 error=None
200/971 article_id=939056 erro

In [32]:
run_label_batch(5000, 6000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 5000 to 6000
Batch size: 1000
Already processed in this batch: 18
Todo: 982
10/982 article_id=188250125120311734 error=None
20/982 article_id=188241017224805169 error=None
30/982 article_id=188230419102924236 error=None
40/982 article_id=1424001 error=None
50/982 article_id=1079204 error=None
60/982 article_id=188240821112106054 error=None
70/982 article_id=188260501174802923 error=None
80/982 article_id=18826050715390473 error=None
90/982 article_id=1197320 error=None
100/982 article_id=188260102001116598 error=None
110/982 article_id=1429122 error=None
120/982 article_id=188240801195127341 error=None
130/982 article_id=1253601 error=None
140/982 article_id=188240731005322419 error=None
150/982 article_id=188240104174124037 error=None
160/982 article_id=188241004163657023 error=None
170/982 article_id=188241016171151564 error=None
180/982 article_id=188250206113248431 error=None
190/982 article_id=1882408090

In [33]:
run_label_batch(6000, 7000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 6000 to 7000
Batch size: 1000
Already processed in this batch: 23
Todo: 977
10/977 article_id=1242701 error=None
20/977 article_id=188260331175847365 error=None
30/977 article_id=188230922215704935 error=None
40/977 article_id=188231116131246417 error=None
50/977 article_id=1382106 error=None
60/977 article_id=1091277 error=None
70/977 article_id=910511 error=None
80/977 article_id=188260418095828416 error=None
90/977 article_id=188241116224737983 error=None
100/977 article_id=970691 error=None
110/977 article_id=188230605163737259 error=None
120/977 article_id=188240302093546337 error=None
125/977 article_id=188241108223728204 error=TimeoutError('The read operation timed out')
130/977 article_id=188230705094541409 error=None
140/977 article_id=188240909102556569 error=None
143/977 article_id=188260329073012567 error=TimeoutError('The read operation timed out')
150/977 article_id=1360000 error=None
160/977 ar

In [36]:
run_label_batch(7000, 8000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 7000 to 8000
Batch size: 1000
Already processed in this batch: 705
Todo: 295
10/295 article_id=188260315223043196 error=None
20/295 article_id=897637 error=None
30/295 article_id=188250609232712135 error=None
40/295 article_id=188241029230449532 error=None
50/295 article_id=1256563 error=None
60/295 article_id=188240506105146739 error=None
70/295 article_id=18826010210403313 error=None
80/295 article_id=1403420 error=None
90/295 article_id=188250409132431325 error=None
100/295 article_id=188251015173251102 error=None
110/295 article_id=188260305225734491 error=None
120/295 article_id=188230630180326254 error=None
130/295 article_id=970529 error=None
140/295 article_id=1102485 error=None
150/295 article_id=188240827104913054 error=None
160/295 article_id=1156322 error=None
170/295 article_id=1278197 error=None
180/295 article_id=188250925225029607 error=None
190/295 article_id=188250116233413091 error=None
200

In [37]:
run_label_batch(8000, 9000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 8000 to 9000
Batch size: 1000
Already processed in this batch: 33
Todo: 967
10/967 article_id=188240224124823873 error=None
20/967 article_id=188251206182034115 error=None
30/967 article_id=188260421105753333 error=None
40/967 article_id=188230925100108365 error=None
50/967 article_id=18826020621494972 error=None
60/967 article_id=880741 error=None
70/967 article_id=188230629163519707 error=None
80/967 article_id=188250512090854405 error=None
90/967 article_id=1117548 error=None
100/967 article_id=188230810085853059 error=None
110/967 article_id=1294406 error=None
120/967 article_id=917689 error=None
130/967 article_id=18823102609460303 error=None
140/967 article_id=974386 error=None
150/967 article_id=188251215182017134 error=None
160/967 article_id=188251026081030992 error=None
170/967 article_id=188230918090042549 error=None
180/967 article_id=828552 error=None
190/967 article_id=1283955 error=None
200/967

In [10]:
run_label_batch(9000, 10000, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 9000 to 10000
Batch size: 1000
Already processed in this batch: 998
Todo: 2


In [11]:
run_label_batch(9000, 9500, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 9000 to 9500
Batch size: 500
Already processed in this batch: 499
Todo: 1


In [15]:
run_label_batch(10000, 10500, run_batch=True)


Output: /Users/buithianhdao/SE365/data/processed/10k_labeled_news.jsonl
Rows: 10000 to 10500
Batch size: 500
Already processed in this batch: 22
Todo: 478
10/478 article_id=188260402145120997 error=None
20/478 article_id=188230411221842214 error=None
30/478 article_id=1024552 error=None
40/478 article_id=18824021322565935 error=None
50/478 article_id=188260130151735162 error=None
60/478 article_id=188240620010719361 error=None
70/478 article_id=851987 error=None
80/478 article_id=188250417164542254 error=None
90/478 article_id=188240316121336671 error=None
100/478 article_id=188230409120117759 error=None
110/478 article_id=188250625173419872 error=None
120/478 article_id=1882404251005284 error=None
130/478 article_id=188260402004502767 error=None
140/478 article_id=1003831 error=None
150/478 article_id=188231128233143633 error=None
160/478 article_id=188241205204807422 error=None
170/478 article_id=188230814155010245 error=None
180/478 article_id=188250217101624342 error=None
190/478 a

KeyboardInterrupt: 

## Inspect Results And Cost Tokens

In [9]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


records = load_jsonl(OUTPUT_PATH)
print("records:", len(records))
if records:
    result_df = pd.DataFrame(records)
    result_df["raw_response_chars"] = result_df["raw_response"].apply(lambda x: len(x) if isinstance(x, str) else 0)
    display(result_df[["article_id", "title", "published_date", "error", "finish_reason", "raw_response_chars"]].head(20))

    usage = pd.json_normalize([r.get("usage", {}) for r in records])
    if not usage.empty:
        print(usage.sum(numeric_only=True).to_string())

    labels = [r.get("label") for r in records if r.get("label")]
    event_counts = {}
    for label in labels:
        for event in label.get("events", []) or []:
            event_type = event.get("event_type")
            event_counts[event_type] = event_counts.get(event_type, 0) + 1
    print("event_counts:")
    print(json.dumps(dict(sorted(event_counts.items(), key=lambda x: str(x[0]))), ensure_ascii=False, indent=2))


records: 1000


,article_id,title,published_date,error,finish_reason,raw_response_chars
0,188231229103257892,"Thủy điện Hủa Na (HNA) ước lãi 217 tỷ đồng, lê...",2023-12-29,None,stop,2695
1,188231229094645055,Hành trình 16 năm và những dấu ấn tiên phong c...,2023-12-29,None,stop,750
2,188231229082554747,"UBCKNN xử phạt Công ty Đại Nam của ông Dũng ""L...",2023-12-29,None,stop,1797
3,188231228162951823,Từng gây chấn động với giá gần 1 triệu đồng/cp...,2023-12-29,None,stop,3582
4,188231228183151294,Góc nhìn CTCK: Rung lắc có thể sớm xuất hiện k...,2023-12-28,None,stop,1831
5,18823122817101572,"Cùng chiều khối ngoại, tự doanh CTCK mua ròng ...",2023-12-28,None,stop,3122
6,188231228165707197,Cổ phiếu nhà Vingroup tiếp tục gây chú ý,2023-12-28,None,stop,5124
7,188231228164957233,Hãng dược hơn trăm tuổi Nhật Bản tung 200 tỷ m...,2023-12-28,None,stop,3003
8,18823122815555081,Khối ngoại tiếp đà mua ròng 330 tỷ đồng trong ...,2023-12-28,None,stop,16297
9,188231228133628241,Cổ phiếu của một công ty vốn nghìn tỷ bị huỷ n...,2023-12-28,None,stop,1762


prompt_tokens                          10531217
completion_tokens                       1426644
total_tokens                           11957861
prompt_cache_hit_tokens                 9181184
prompt_cache_miss_tokens                1350033
prompt_tokens_details.cached_tokens     9181184
event_counts:
{
  "declare_bankruptcy": 6,
  "dividend": 270,
  "earnings_report": 399,
  "equity_freeze": 7,
  "equity_overweight": 236,
  "equity_repurchase": 9,
  "equity_underweight": 311,
  "lawsuit": 20,
  "legal_penalty": 110,
  "macro_indicator": 145,
  "merge_org": 12,
  "other": 36,
  "personnel_change": 76,
  "price_movement": 999,
  "share_issuance": 209,
  "transfer_money": 47,
  "transfer_ownership": 60
}


## Inspect Raw Response For Failed Articles

Sau khi batch chạy, dùng helper này để in `raw_response` của một `article_id`, nhất là các dòng bị `JSONDecodeError`.

In [11]:
import re


def show_raw_response(article_id: str, *, full: bool = False, context_chars: int = 1200) -> None:
    records = load_jsonl(OUTPUT_PATH)
    matches = [r for r in records if str(r.get("article_id")) == str(article_id)]
    print("matches:", len(matches))
    if not matches:
        return

    for idx, record in enumerate(matches, start=1):
        raw = record.get("raw_response")
        error = record.get("error")
        print("\n--- match", idx, "---")
        print("title:", record.get("title"))
        print("error:", error)
        print("finish_reason:", record.get("finish_reason"))
        print("usage:", record.get("usage"))
        print("raw_response_chars:", len(raw) if isinstance(raw, str) else None)

        if not isinstance(raw, str):
            print("raw_response is None. Record này có thể được tạo trước khi sửa notebook; hãy chạy lại batch cho article này.")
            continue

        if full:
            print(raw)
            continue

        match = re.search(r"char (\d+)", str(error))
        if match:
            pos = int(match.group(1))
            start = max(0, pos - context_chars)
            end = min(len(raw), pos + context_chars)
            print(f"showing raw_response[{start}:{end}] around char {pos}:")
            print(raw[start:end])
        else:
            print(f"showing raw_response[:{context_chars}]:")
            print(raw[:context_chars])


# Example:
show_raw_response("188231220091258805")
show_raw_response("188231220091258805", full=True)


matches: 1

--- match 1 ---
title: Lịch sự kiện và tin vắn chứng khoán ngày 20/12
error: None
finish_reason: stop
usage: {'prompt_tokens': 10496, 'completion_tokens': 4903, 'total_tokens': 15399, 'prompt_tokens_details': {'cached_tokens': 10368}, 'prompt_cache_hit_tokens': 10368, 'prompt_cache_miss_tokens': 128}
raw_response_chars: 13559
showing raw_response[:1200]:
{
  "doc_id": "188231220091258805",
  "language": "vi",
  "main_topic": "ownership_change",
  "entities": [
    {"name": "PYN ELITE FUND (NON-UCITS)", "type": "Organization", "ticker": null, "role_in_article": "subject"},
    {"name": "Công ty Cổ phần Dịch vụ Hàng hóa Sài Gòn", "type": "Company", "ticker": "SCS", "role_in_article": "mentioned"},
    {"name": "CT TNHH Sài Gòn 3 Jean", "type": "Organization", "ticker": null, "role_in_article": "subject"},
    {"name": "Công ty Cổ phần Pin Ắc quy miền Nam", "type": "Company", "ticker": "PAC", "role_in_article": "mentioned"},
    {"name": "Nguyễn Thị Nga", "type": "Person", "ti